In [7]:
# Imports and setup
import fsspec
import pyarrow.parquet as pq
import pyarrow as pa
import pyarrow.compute as pc

import os
import time
from datetime import datetime

print("All libraries imported successfully!")

All libraries imported successfully!


# Remote Schema expolaration

In [1]:
URLS = {
    "Blocks": "https://bsky-data.leobalduf.com/blocks.parquet",
    "Follows": "https://bsky-data.leobalduf.com/follows.parquet",
    "Likes": "https://bsky-data.leobalduf.com/likes.parquet",
    "Posts": "https://bsky-data.leobalduf.com/posts.parquet",
    "Profiles": "https://bsky-data.leobalduf.com/profiles.parquet"
}

In [ ]:
for name, url in URLS.items():
    with fsspec.open(url, "rb") as f:
        parquet_file = pq.ParquetFile(f)
        print(f"{name} Database")
        print("Columns:", parquet_file.schema.names)
        print("Number of rows:", parquet_file.metadata.num_rows)
        print("Number of row groups:", parquet_file.metadata.num_row_groups)

In [2]:
def get_column_metadata(URL, column_names=None, detailed=False):
    
    with fsspec.open(URL, "rb") as f:
        parquet_file = pq.ParquetFile(f)
        metadata = parquet_file.metadata
        schema = parquet_file.schema_arrow
        
        # Handle column selection
        if column_names is None:
            column_names = schema.names
        elif isinstance(column_names, str):
            column_names = [column_names]
        
        # Validate columns exist
        valid_columns = []
        for col_name in column_names:
            if col_name in schema.names:
                valid_columns.append(col_name)
            else:
                print(f"Warning: Column '{col_name}' not found in schema. Available columns: {schema.names}")
        
        if not valid_columns:
            print("No valid columns to analyze")
            return
        
        mode = "DETAILED" if detailed else "OVERVIEW"
        print(f"=== {mode} FOR {len(valid_columns)} COLUMN(S) ===")
        
        # Process each column
        for column_name in valid_columns:
            col_idx = schema.names.index(column_name)
            col_type = schema.field(column_name).type
            
            if detailed:
                print(f"\n{'='*50}")
                print(f"DETAILED METADATA FOR '{column_name}'")
                print(f"{'='*50}")
            else:
                print(f"\n{column_name}: {col_type}")
            
            # Initialize aggregators
            total_nulls = 0
            total_values = 0
            has_min_max = False
            global_min = None
            global_max = None
            has_distinct_count = False
            distinct_count = None
            
            # Aggregate across all row groups
            for rg_idx in range(metadata.num_row_groups):
                rg_metadata = metadata.row_group(rg_idx)
                col_metadata = rg_metadata.column(col_idx)
                rg_num_rows = rg_metadata.num_rows
                
                if col_metadata.statistics:
                    stats = col_metadata.statistics
                    
                    # Aggregate null count
                    if stats.has_null_count:
                        total_nulls += stats.null_count
                    
                    # Aggregate min/max values
                    if stats.has_min_max:
                        has_min_max = True
                        try:
                            current_min = stats.min
                            current_max = stats.max
                            
                            # Handle different data types for comparison
                            if pa.types.is_timestamp(col_type):
                                current_min = pa.scalar(current_min).cast(col_type).as_py()
                                current_max = pa.scalar(current_max).cast(col_type).as_py()
                            
                            # Update global min/max
                            if global_min is None or current_min < global_min:
                                global_min = current_min
                            if global_max is None or current_max > global_max:
                                global_max = current_max
                                
                        except (OverflowError, Exception):
                            # If we can't compare, just take the first available values
                            if global_min is None:
                                global_min = stats.min
                                global_max = stats.max
                    
                    # Handle distinct count (take max across row groups as approximation)
                    if stats.has_distinct_count:
                        has_distinct_count = True
                        if distinct_count is None or stats.distinct_count > distinct_count:
                            distinct_count = stats.distinct_count
                
                total_values += rg_num_rows
            
            # Calculate percentages
            null_percentage = (total_nulls / total_values) * 100 if total_values > 0 else 0
            non_null_count = total_values - total_nulls
            
            # Display results based on detail level
            if detailed:
                print(f"Data type: {col_type}")
                print(f"Total rows: {total_values:,}")
                print(f"Null count: {total_nulls:,} ({null_percentage:.2f}%)")
                print(f"Non-null count: {non_null_count:,}")
                
                if has_min_max:
                    print(f"\nValue Range:")
                    try:
                        if pa.types.is_timestamp(col_type):
                            print(f"  Min: {global_min}")
                            print(f"  Max: {global_max}")
                        elif pa.types.is_string(col_type) or pa.types.is_large_string(col_type):
                            min_str = str(global_min)
                            max_str = str(global_max)
                            if len(min_str) > 50:
                                min_str = min_str[:47] + "..."
                            if len(max_str) > 50:
                                max_str = max_str[:47] + "..."
                            print(f"  Min: {min_str}")
                            print(f"  Max: {max_str}")
                        else:
                            print(f"  Min: {global_min}")
                            print(f"  Max: {global_max}")
                    except Exception as e:
                        print(f"  Min: <error displaying: {e}>")
                        print(f"  Max: <error displaying: {e}>")
                else:
                    print(f"\nValue Range: No min/max statistics available")
                
                if has_distinct_count:
                    print(f"Distinct values: {distinct_count:,}")
                else:
                    print(f"Distinct values: No distinct count statistics available")
                    
            else:
                # Quick overview mode
                print(f"  Total rows: {total_values:,}")
                print(f"  Null count: {total_nulls:,} ({null_percentage:.1f}%)")
                print(f"  Non-null: {non_null_count:,}")
                
                if has_min_max:
                    try:
                        if pa.types.is_timestamp(col_type):
                            print(f"  Value range: {global_min} to {global_max}")
                        elif pa.types.is_string(col_type) or pa.types.is_large_string(col_type):
                            min_str = str(global_min)[:20] + "..." if len(str(global_min)) > 20 else str(global_min)
                            max_str = str(global_max)[:20] + "..." if len(str(global_max)) > 20 else str(global_max)
                            print(f"  Value range: '{min_str}' to '{max_str}'")
                        else:
                            print(f"  Value range: {global_min} to {global_max}")
                    except Exception:
                        print(f"  Value range: Available (display error)")
                else:
                    print(f"  Value range: No statistics available")

In [ ]:
get_column_metadata(URLS["Posts"], ["did_id", "created_at"], detailed=True)

=== DETAILED FOR 2 COLUMN(S) ===

DETAILED METADATA FOR 'did_id'
Data type: int64
Total rows: 1,290,686,413
Null count: 0 (0.00%)
Non-null count: 1,290,686,413

Value Range:
  Min: 3
  Max: 34251847
Distinct values: No distinct count statistics available

DETAILED METADATA FOR 'created_at'
Data type: timestamp[us, tz=UTC]
Total rows: 1,290,686,413
Null count: 76 (0.00%)
Non-null count: 1,290,686,337

Value Range:
  Min: 0001-01-01 00:00:00+00:00
  Max: 9999-12-31 23:59:59.999000+00:00
Distinct values: No distinct count statistics available


In [ ]:
get_column_metadata(URLS["Profiles"], ["did_id", "created_at", "joined_via_starter_pack"], detailed=True)

=== DETAILED FOR 3 COLUMN(S) ===

DETAILED METADATA FOR 'did_id'
Data type: int64
Total rows: 32,170,299
Null count: 0 (0.00%)
Non-null count: 32,170,299

Value Range:
  Min: 1
  Max: 34251851
Distinct values: No distinct count statistics available

DETAILED METADATA FOR 'created_at'
Data type: timestamp[us, tz=UTC]
Total rows: 32,170,299
Null count: 4,247,596 (13.20%)
Non-null count: 27,922,703

Value Range:
  Min: 0081-11-15 02:48:08.855000+00:00
  Max: 2954-08-31 07:36:16.393000+00:00
Distinct values: No distinct count statistics available

DETAILED METADATA FOR 'joined_via_starter_pack'
Data type: extension<arrow.json>
Total rows: 32,170,299
Null count: 1,488,624 (4.63%)
Non-null count: 30,681,675

Value Range:
  Min: application/octet-stream
  Max: text/html
Distinct values: 5


# Duck db trying. 
Note: From the EDA we know that did_id and subject_id are never null, so we don't include them in the cleaning. 

In [22]:
MAX_DATE = '2025-05-14'
MIN_DATE = '2021-01-01'

In [ ]:
import duckdb

# DuckDB-based cleaning helper function (with optional column projection)
def clean_parquet_with_duckdb(input_path, output_path, where_clause=None, columns=None, label=None, verbose=True):
    """
    Clean a parquet file using DuckDB and write the result as a parquet file (ZSTD compression).

    Args:
        input_path: path to the input parquet file
        output_path: path where the cleaned parquet will be written
        where_clause: optional SQL WHERE clause (string) to filter rows (without the WHERE keyword)
        columns: optional list of column names to select (e.g. ['did_id','created_at'])
        label: optional label for printed messages
        verbose: if True, print progress and statistics

    Returns:
        dict with statistics: elapsed_time, total_rows, filtered_rows, retention_rate, input_size_gb, output_size_gb
    """
    if label is None:
        label = os.path.basename(input_path)

    if verbose:
        print(f"Cleaning (DuckDB) → {label}")

    start_time = time.time()
    conn = duckdb.connect()
    try:
        # Count total rows in the source file (no projection here; we need the full count)
        total_rows = conn.execute(f"SELECT COUNT(*) FROM read_parquet('{input_path}')").fetchone()[0]

        # Build the projection (columns) SQL
        if columns:
            # Simple join, assume caller provides valid column names
            cols_sql = ', '.join(columns)
        else:
            cols_sql = '*'

        # Build the SELECT SQL (with optional WHERE)
        if where_clause:
            select_sql = f"SELECT {cols_sql} FROM read_parquet('{input_path}') WHERE {where_clause}"
        else:
            select_sql = f"SELECT {cols_sql} FROM read_parquet('{input_path}')"

        # Use COPY to write a compressed parquet file efficiently
        copy_sql = f"COPY ({select_sql}) TO '{output_path}' (FORMAT PARQUET, COMPRESSION ZSTD);"
        conn.execute(copy_sql)

        # Count rows written
        filtered_rows = conn.execute(f"SELECT COUNT(*) FROM read_parquet('{output_path}')").fetchone()[0]
    finally:
        conn.close()

    elapsed = time.time() - start_time
    input_size = os.path.getsize(input_path) if os.path.exists(input_path) else 0
    output_size = os.path.getsize(output_path) if os.path.exists(output_path) else 0
    retention_rate = (filtered_rows / total_rows * 100) if total_rows > 0 else 0

    if verbose:
        print("\n✅ DuckDB cleaning completed")
        print(f"Time: {elapsed:.2f} sec")
        print(f"Rows: {total_rows:,} → {filtered_rows:,} ({retention_rate:.1f}% kept)")
        print(f"File size: {input_size / (1024**3):.3f} GB → {output_size / (1024**3):.3f} GB")
        print(f"Output: {output_path}")

    return {
        'elapsed_time': elapsed,
        'total_rows': total_rows,
        'filtered_rows': filtered_rows,
        'retention_rate': retention_rate,
        'input_size_gb': input_size / (1024**3),
        'output_size_gb': output_size / (1024**3),
    }

In [ ]:
# Chunk_0 Posts
in_path = "../data/raw/chunk_0_posts.parquet"
out_path = "../data/posting/cleaned/chunk_0_posts.parquet"

cols = ['did_id', 'created_at']
where = "created_at IS NOT NULL AND created_at <= '{MAX_DATE}' AND created_at >= '{MIN_DATE}'".format(MAX_DATE=MAX_DATE, MIN_DATE=MIN_DATE)

stats_posts = clean_parquet_with_duckdb(in_path, out_path, where_clause=where, columns=cols, label="chunk_0_posts (duckdb, projected)")
print(stats_posts)

Cleaning (DuckDB) → chunk_0_posts (duckdb, projected)

✅ DuckDB cleaning completed
Time: 1.20 sec
Rows: 5,000,000 → 4,992,762 (99.9% kept)
File size: 0.260 GB → 0.031 GB
Output: ../data/posting/cleaned/chunk_0_posts_duckdb.parquet
{'elapsed_time': 1.1975719928741455, 'total_rows': 5000000, 'filtered_rows': 4992762, 'retention_rate': 99.85524, 'input_size_gb': 0.25970588624477386, 'output_size_gb': 0.031201929785311222}

✅ DuckDB cleaning completed
Time: 1.20 sec
Rows: 5,000,000 → 4,992,762 (99.9% kept)
File size: 0.260 GB → 0.031 GB
Output: ../data/posting/cleaned/chunk_0_posts_duckdb.parquet
{'elapsed_time': 1.1975719928741455, 'total_rows': 5000000, 'filtered_rows': 4992762, 'retention_rate': 99.85524, 'input_size_gb': 0.25970588624477386, 'output_size_gb': 0.031201929785311222}


In [ ]:
# Profiles
in_path = "../data/raw/profiles.parquet"
out_path = "../data/posting/cleaned/profiles.parquet"

cols = ['did_id', 'created_at']
where = "did_id IS NOT NULL AND created_at IS NOT NULL AND created_at <= '{MAX_DATE}' AND created_at >= '{MIN_DATE}'".format(MAX_DATE=MAX_DATE, MIN_DATE=MIN_DATE)

stats = clean_parquet_with_duckdb(in_path, out_path, where_clause=where, columns=cols, label="profiles (duckdb, projected)")

Cleaning (DuckDB) → profiles (duckdb, projected)

✅ DuckDB cleaning completed
Time: 4.36 sec
Rows: 32,170,299 → 27,922,673 (86.8% kept)
File size: 1.772 GB → 0.201 GB
Output: ../data/posting/cleaned/profiles_duckdb.parquet

✅ DuckDB cleaning completed
Time: 4.36 sec
Rows: 32,170,299 → 27,922,673 (86.8% kept)
File size: 1.772 GB → 0.201 GB
Output: ../data/posting/cleaned/profiles_duckdb.parquet


In [ ]:
# Blocks
in_path = "../data/raw/blocks.parquet"
out_path = "../data/posting/cleaned/blocks.parquet"

# Project only the two columns we care about to reduce IO
cols = ['did_id', 'created_at', 'subject_id']
where = "created_at IS NOT NULL AND created_at <= '{MAX_DATE}' AND created_at >= '{MIN_DATE}'".format(MAX_DATE=MAX_DATE, MIN_DATE=MIN_DATE)

stats_blocks = clean_parquet_with_duckdb(in_path, out_path, where_clause=where, columns=cols, label="blocks (duckdb, projected)")
print(stats_blocks)

Cleaning (DuckDB) → blocks (duckdb, projected)

✅ DuckDB cleaning completed
Time: 25.04 sec
Rows: 120,088,510 → 120,084,926 (100.0% kept)
File size: 1.489 GB → 1.007 GB
Output: ../data/posting/cleaned/blocks_duckdb.parquet
{'elapsed_time': 25.04248857498169, 'total_rows': 120088510, 'filtered_rows': 120084926, 'retention_rate': 99.99701553462526, 'input_size_gb': 1.4892841363325715, 'output_size_gb': 1.0068980706855655}

✅ DuckDB cleaning completed
Time: 25.04 sec
Rows: 120,088,510 → 120,084,926 (100.0% kept)
File size: 1.489 GB → 1.007 GB
Output: ../data/posting/cleaned/blocks_duckdb.parquet
{'elapsed_time': 25.04248857498169, 'total_rows': 120088510, 'filtered_rows': 120084926, 'retention_rate': 99.99701553462526, 'input_size_gb': 1.4892841363325715, 'output_size_gb': 1.0068980706855655}


In [ ]:
# Follows
in_path = "../data/raw/follows.parquet"
out_path = "../data/posting/cleaned/follows.parquet"

cols = ['did_id', 'created_at', 'subject_id']
where = "created_at IS NOT NULL AND created_at <= '{MAX_DATE}' AND created_at >= '{MIN_DATE}'".format(MAX_DATE=MAX_DATE, MIN_DATE=MIN_DATE)

stats_follows = clean_parquet_with_duckdb(in_path, out_path, where_clause=where, columns=cols, label="follows (duckdb, projected)")
print(stats_follows)

Cleaning (DuckDB) → follows (duckdb, projected)

✅ DuckDB cleaning completed
Time: 411.93 sec
Rows: 2,156,778,700 → 2,156,734,110 (100.0% kept)
File size: 23.174 GB → 15.186 GB
Output: ../data/posting/cleaned/follows_duckdb.parquet
{'elapsed_time': 411.92923402786255, 'total_rows': 2156778700, 'filtered_rows': 2156734110, 'retention_rate': 99.99793256489411, 'input_size_gb': 23.17432524636388, 'output_size_gb': 15.185829509049654}

✅ DuckDB cleaning completed
Time: 411.93 sec
Rows: 2,156,778,700 → 2,156,734,110 (100.0% kept)
File size: 23.174 GB → 15.186 GB
Output: ../data/posting/cleaned/follows_duckdb.parquet
{'elapsed_time': 411.92923402786255, 'total_rows': 2156778700, 'filtered_rows': 2156734110, 'retention_rate': 99.99793256489411, 'input_size_gb': 23.17432524636388, 'output_size_gb': 15.185829509049654}
